In [ ]:
#!pip install transformers datasets pandas torch scikit-learn

  Using cached transformers-5.4.0-py3-none-any.whl.metadata (32 kB)
  Using cached datasets-4.8.4-py3-none-any.whl.metadata (19 kB)
  Using cached huggingface_hub-1.8.0-py3-none-any.whl.metadata (13 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached typer-0.24.1-py3-none-any.whl.metadata (16 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached aiohttp-3.13.5-cp311-cp311-win_amd64.whl.metadata (8.4 kB)
  Using cached rich-14.3.3-py3-none-any.whl.metadata (18 kB)
Using cached transformers-5.4.0-py3-none-any.whl (10.1 MB)
Using cached datasets-4.8.4-py3-none-any.whl (526 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached huggingface_hub-1.8.0-py3-none-any.whl (625 kB)
Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl (2.7 MB)
Using cached typer-0.24.1-py3-none-any.whl (56 kB)
Using cached aiohttp-3.13.5-cp311-cp311-win_amd64.whl (462 kB)
Using cached rich-14.3.3-py3-none-any.whl (310 kB)



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import re
from string import Template

c:\Users\Fei\Project\YouTubeCategories-4NL3\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
training_data = pd.read_csv("./codabench/app/input_data/training_data.csv")
training_label = pd.read_csv("./codabench/app/input_data/training_label.csv")
combined_training = pd.concat([training_data, training_label], axis=1)
combined_training.head()
sample = combined_training.sample(n=200)

In [3]:
test_data = pd.read_csv("./codabench/app/input_data/testing_data.csv")
test_label = pd.read_csv("./codabench/app/reference_data/testing_label.csv")
combined_test= pd.concat([test_data, test_label], axis=1)
combined_test.head()
print(len(combined_test))

279


In [4]:
MODEL_NAME = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

num_to_category = {
    -1: "None",
    1: "Food & Cooking",
    2: "Video Games",
    3: "Sports & Games",
    4: "Music",
    5: "Education",
    6: "Shopping",
    7: "Vehicles",
    8: "Lifestyle",
    9: "News",
    10: "Health",
    11: "Business",
    12: "Digital Media"
}

Loading weights: 100%|██████████| 558/558 [00:00<00:00, 4024.44it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [5]:
def predict_label(prompt_text):
    inputs = tokenizer(prompt_text, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=10)
    pred_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if pred_text.startswith(prompt_text):
        pred_text = pred_text[len(prompt_text)].strip()
    match = re.search(r"\b(1[0-2]|[1-9])\b", pred_text)
    if match:
        return int(match.group(1))
    else:
        return -1

In [6]:
def test_prompt(prompt, dataset):
    correct = 0
    t_1 = Template(prompt)
    for _, row in dataset.iterrows():
        title = row["video_title"]
        description = row["video_description"]
        label = row["category"]
        prompt_text = t_1.substitute(title=title, description=description)
        pred = predict_label(prompt_text)
        pred_cat = num_to_category[pred]
        if pred_cat == label:
            correct += 1
    acc = correct / len(dataset)
    return acc


In [9]:
prompt_1 =  """You are a topic classifier. You will be given a youtube title and it's description. You must categorize it as one of the following categories: 
    1: Food & Cooking
    2: Video Games
    3: Sports & Games
    4: Music
    5: Education
    6: Shopping
    7: Vehicles
    8: Lifestyle
    9: News
    10: Health
    11: Business
    12: Digital Media
    The task is to answer the classification question, "What is the main category of this YouTube video?". You can achieve this by reading both the title and description in order to determine the most probably category. If a video title and description don't align, place more emphasize on the title.
    Answer only with the corresponding number of the category names.
    Here is the title: $title
    Here is the description: $description
    """

    

In [14]:
acc_1 = test_prompt(prompt_1, sample)
print(acc_1)

0.33


In [10]:
prompt_2 =  """Given a youtube title and it's description. You must categorize it as one of the following categories, with corresponding topics that fit in the category: 
    1: Food & Cooking: Recipe, Cooking Technique
    2: Video Games: Gameplay, Game Reviews, Streaming Highlights
    3: Sports & Games: Sport Games, Commentary, Board Games, Training Specific to Sport
    4: Music: Music Videos, Karaoke Videos
    5. Education: School Coursework, Math, General Knowledge Videos, Documentary
    6: Shopping: Product Reviews, Shopping Hauls
    7: Vehicles: Car Shows, Racing
    8: Lifestyle: Traveling, Day-in-the-life, Vlogs
    9: News: Politics, Breaking News, Current Events
    10: Health & Fitness: Gym, Workout Plans, Diet
    11: Business: Investing, Economy, Entrepreneurship, Finance
    12: Digital Media: Movies, Animated Shorts, Fan Fiction, Memes

    The task is to answer the classification question, "What is the main category of this YouTube video?". You can achieve this by reading both the title and description in order to determine the most probably category. If a video title and description don't align, place more emphasize on the title.
    Answer only with the corresponding number of the category names.
    Here is the title: $title
    Here is the description: $description
    """

In [ ]:
acc_2 = test_prompt(prompt_2, sample)
print(acc_2)  

Token indices sequence length is longer than the specified maximum sequence length for this model (517 > 512). Running this sequence through the model will result in indexing errors


0.415


In [11]:
prompt_3 =  """Given a youtube title. You must categorize it as one of the following categories, with corresponding topics that fit in the category: 
    1: Food & Cooking: Recipe, Cooking Technique
    2: Video Games: Gameplay, Game Reviews, Streaming Highlights
    3: Sports & Games: Sport Games, Commentary, Board Games, Training Specific to Sport
    4: Music: Music Videos, Karaoke Videos
    5. Education: School Coursework, Math, General Knowledge Videos, Documentary
    6: Shopping: Product Reviews, Shopping Hauls
    7: Vehicles: Car Shows, Racing
    8: Lifestyle: Traveling, Day-in-the-life, Vlogs
    9: News: Politics, Breaking News, Current Events
    10: Health & Fitness: Gym, Workout Plans, Diet
    11: Business: Investing, Economy, Entrepreneurship, Finance
    12: Digital Media: Movies, Animated Shorts, Fan Fiction, Memes

    The task is to answer the classification question, "What is the main category of this YouTube video?". You can achieve this by reading the tile.
    Answer only with the corresponding number of the category names.
    Here is the title: $title
    """

In [16]:
acc_3 = test_prompt(prompt_3, sample)
print(acc_3)

0.365


In [ ]:
best_prompt = prompt_2
def get_predictions(prompt, dataset):
    res = []
    t = Template(prompt)

    for _, row in dataset.iterrows():
        title = row["video_title"]
        description = row["video_description"]

        prompt_text = t.substitute(title=title, description=description)
        pred = predict_label(prompt_text)
        pred_cat = num_to_category[pred]

        res.append({
            "predicted_label": pred_cat
        })

    return pd.DataFrame(res)


In [20]:
predicted_labels = get_predictions(best_prompt, combined_test)

In [30]:
from sklearn.metrics import classification_report
cr = classification_report(combined_test["category"], predicted_labels)
cr_dict = classification_report(combined_test["category"], predicted_labels, output_dict=True)

c:\Users\Fei\Project\YouTubeCategories-4NL3\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Fei\Project\YouTubeCategories-4NL3\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Fei\Project\YouTubeCategories-4NL3\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.

In [31]:
print(cr)

                precision    recall  f1-score   support

      Business       0.60      0.75      0.67         8
 Digital Media       0.47      0.30      0.37        23
     Education       0.81      0.39      0.53        76
Food & Cooking       0.59      0.90      0.71        30
        Health       0.00      0.00      0.00         3
     Lifestyle       0.43      0.22      0.29        45
         Music       0.09      0.57      0.15         7
          News       0.44      0.40      0.42        20
          None       0.00      0.00      0.00         0
      Shopping       0.46      0.69      0.55        16
Sports & Games       0.50      0.45      0.47        20
       Vehicle       0.00      0.00      0.00         6
      Vehicles       0.00      0.00      0.00         0
   Video Games       0.45      0.60      0.52        25

      accuracy                           0.46       279
     macro avg       0.35      0.38      0.33       279
  weighted avg       0.55      0.46      0.46 

In [32]:
print(cr_dict)

{'Business': {'precision': 0.6, 'recall': 0.75, 'f1-score': 0.6666666666666666, 'support': 8.0}, 'Digital Media': {'precision': 0.4666666666666667, 'recall': 0.30434782608695654, 'f1-score': 0.3684210526315789, 'support': 23.0}, 'Education': {'precision': 0.8108108108108109, 'recall': 0.39473684210526316, 'f1-score': 0.5309734513274337, 'support': 76.0}, 'Food & Cooking': {'precision': 0.5869565217391305, 'recall': 0.9, 'f1-score': 0.7105263157894737, 'support': 30.0}, 'Health': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 3.0}, 'Lifestyle': {'precision': 0.43478260869565216, 'recall': 0.2222222222222222, 'f1-score': 0.29411764705882354, 'support': 45.0}, 'Music': {'precision': 0.08695652173913043, 'recall': 0.5714285714285714, 'f1-score': 0.1509433962264151, 'support': 7.0}, 'News': {'precision': 0.4444444444444444, 'recall': 0.4, 'f1-score': 0.42105263157894735, 'support': 20.0}, 'None': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 0.0}, 'Shopping': 